In [1]:
!pip install -q google-genai pandas

In [4]:
from google.colab import userdata
from google import genai
import pandas as pd
import time

MODEL = "gemini-3.1-flash-lite"

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options={"timeout": 120000}
)

print("Gemini API connected successfully!")
print("Model:", MODEL)

Gemini API connected successfully!
Model: gemini-3.1-flash-lite


In [5]:
import time

SYSTEM = "You are a course assistant for Generative AI. Answer briefly and cite the course handout."

CONTEXT = "Module III covers embeddings, chunking, MongoDB, Chroma and RAG evaluation with Hit Rate@k and MRR. " * 8

questions = [
    "What is RAG?",
    "Explain MRR with an example.",
    "రాగ్ అంటే ఏమిటి?",
    "आरएजी क्या है?"
]

rows = []

for q in questions:
    print("Processing:", q)

    for attempt in range(3):
        try:
            r = client.models.generate_content(
                model=MODEL,
                contents=SYSTEM + "\n" + CONTEXT + "\nQuestion: " + q,
                config={"max_output_tokens": 128}
            )

            rows.append({
                "question": q,
                "input_tokens": r.usage_metadata.prompt_token_count,
                "output_tokens": r.usage_metadata.candidates_token_count
            })

            print("Completed successfully!")
            break

        except Exception as e:
            print("Attempt", attempt + 1, "failed:", e)
            if attempt < 2:
                time.sleep(5)

    time.sleep(3)

sample = pd.DataFrame(rows)

print("\nTOKEN USAGE REPORT")
display(sample)

Processing: What is RAG?
Completed successfully!
Processing: Explain MRR with an example.
Completed successfully!
Processing: రాగ్ అంటే ఏమిటి?
Completed successfully!
Processing: आरएजी क्या है?
Completed successfully!

TOKEN USAGE REPORT


,question,input_tokens,output_tokens
0,What is RAG?,221,66
1,Explain MRR with an example.,223,124
2,రాగ్ అంటే ఏమిటి?,222,72
3,आरएजी क्या है?,222,68


In [6]:
USERS_PER_DAY = 1000

QUESTIONS_PER_USER = 5

PEAK_HOUR_SHARE = 0.20

PRICE_IN = 0.10
PRICE_OUT = 0.40

RPM_LIMIT = 15
RPD_LIMIT = 1000

CONTEXT_LIMIT = 1_000_000

print("Budget assumptions configured successfully!")

print("Users per day:", USERS_PER_DAY)
print("Questions per user:", QUESTIONS_PER_USER)
print("Peak hour share:", PEAK_HOUR_SHARE)
print("Input price:", PRICE_IN)
print("Output price:", PRICE_OUT)
print("RPM limit:", RPM_LIMIT)
print("RPD limit:", RPD_LIMIT)

Budget assumptions configured successfully!
Users per day: 1000
Questions per user: 5
Peak hour share: 0.2
Input price: 0.1
Output price: 0.4
RPM limit: 15
RPD limit: 1000


In [7]:
avg_in = sample.input_tokens.mean()
avg_out = sample.output_tokens.mean()

req_day = USERS_PER_DAY * QUESTIONS_PER_USER

cost_req = (avg_in * PRICE_IN + avg_out * PRICE_OUT) / 1e6

peak_rpm = req_day * PEAK_HOUR_SHARE / 60

budget = {
    "avg input tokens / request": round(avg_in),
    "avg output tokens / request": round(avg_out),

    "requests / day": req_day,

    "tokens / day": round(req_day * (avg_in + avg_out)),

    "cost / request (USD)": round(cost_req, 6),

    "cost / day (USD)": round(cost_req * req_day, 2),

    "cost / month (USD)": round(cost_req * req_day * 30, 2),

    "cost / month (INR @ 84)": round(cost_req * req_day * 30 * 84),

    "peak requests / minute": round(peak_rpm, 1),

    "fits free-tier RPM?": peak_rpm <= RPM_LIMIT,

    "fits free-tier RPD?": req_day <= RPD_LIMIT,

    "context used per request": f"{avg_in / CONTEXT_LIMIT:.4%}"
}

print("Token economics calculated successfully!")

Token economics calculated successfully!


In [8]:
print("\n========== TOKEN ECONOMICS BUDGET REPORT ==========\n")

for k, v in budget.items():
    print(f"{k:28s}: {v}")

print("\n===================================================")

print("\nFINAL DECISION:")

if peak_rpm <= RPM_LIMIT and req_day <= RPD_LIMIT:
    print("The estimated workload fits the configured RPM and RPD limits.")
else:
    print("The estimated workload exceeds the configured limits.")
    print("Consider caching, a paid tier, or a request queue.")

print("\nExperiment completed successfully!")


========== TOKEN ECONOMICS BUDGET REPORT ==========

avg input tokens / request  : 222
avg output tokens / request : 82
requests / day              : 5000
tokens / day                : 1522500
cost / request (USD)        : 5.5e-05
cost / day (USD)            : 0.28
cost / month (USD)          : 8.28
cost / month (INR @ 84)     : 696
peak requests / minute      : 16.7
fits free-tier RPM?         : False
fits free-tier RPD?         : False
context used per request    : 0.0222%


FINAL DECISION:
The estimated workload exceeds the configured limits.
Consider caching, a paid tier, or a request queue.

Experiment completed successfully!
